# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedkhaled600/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

***Answer:***

This is a **scoring / ranking task, built on top of a binary classification model.**

The underlying decision is "which pages should an editor look at first?" — a "which ones first?"
question, which maps to ranking/scoring per the framing guide. But the score itself needs a
learnable target, so under the hood this is a binary classifier: predict whether a page is
currently declining, then use the model's predicted probability as the priority score that
produces the ranked queue. Pure clustering doesn't fit (I already have a labeled outcome to
predict, not just "what groups exist"), and pure classification alone doesn't fit either, since
the final deliverable an editor needs is an ordered list with a cutoff (top 20, top 50), not just
a yes/no per page — hence "scoring," not "classification," as the top-level frame.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

***Answer:***

**Target:** `is_declining_label` — 1 if `trend_direction == "down"`, else 0. Per the data
dictionary, `trend_direction == "down"` means the page's impressions fell more than 20% comparing
the last 30 days to the previous 30 days.

**Where the label comes from:** this is an **observed outcome**, not a rule I invented — it's a
real, measured change in traffic (`impressions_last_30d` vs `impressions_prev_30d`), not a
subjective judgment call about which pages "look" like they need work. The one honest caveat: the
±20% cutoff that turns a continuous change (`trend_pct`) into a yes/no label *was* set by the data
pipeline, not discovered by me — so I'm treating "declining" as a threshold on a real measurement,
not as ground truth about business importance. Because the label is built directly from
`trend_pct`/`trend_direction`, those two columns can never be used as model features — that would
be leakage (the model would just be learning to reverse the label's own definition).

In [2]:
import os
if not os.path.exists("flyrank-internship-ml"):
    !git clone --quiet https://github.com/mohamedkhaled600/flyrank-internship-ml.git
%cd flyrank-internship-ml/work/notebooks

/content/flyrank-internship-ml/work/notebooks


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df))
print(df["is_declining_label"].value_counts())
print()
print("Base rate (share declining):", round(100 * df["is_declining_label"].mean(), 1), "%")


Rows: 30000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (share declining): 54.2 %


## 3. Success metric

*One metric you can defend. What number means 'good'?*

***Answer:***

**Metric: precision@K** (with recall and average precision as supporting numbers), evaluated
against the 54.2% base rate — because the editor only ever works through a fixed-size queue (say,
the top 20 or top 50 pages this sprint), what matters is "of the pages I actually send someone to
review, how many were truly worth reviewing?" — not overall accuracy, which a model could win by
just predicting the majority class. I'll also track plain ROC-AUC on the underlying classifier as
a model-quality check, but precision@K is the number I'd defend to a content lead, because it's
denominated in the same unit as their week: hours spent on pages that were actually worth it.

**What "good" means, concretely:** precision@20 clearly above 54.2% (better than randomly grabbing
20 pages) — ideally 75-85%+, since that's the difference between "half your review time is wasted"
and "most of it lands on a real problem."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

***Answer:***

**One row = one content page** (`content_id`), pseudonymized, with its trailing-90-day
performance and content attributes attached. Below is the lane's working slice: the identifying
columns, the content attributes I'd use as features, and the label — with the label-source
columns (`trend_pct`, `trend_direction`) kept visible only so I can show where the label came
from, never as inputs to a model.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_cols = [
    "content_id", "client_id",
    # content attributes (candidate features)
    "content_type", "main_intent", "word_count", "content_age_days", "age_tier",
    "days_since_last_update", "freshness_tier", "avg_position", "position_tier",
    "search_volume", "competition_level", "engagement_rate", "scroll_rate",
    # label + its source (source columns shown for transparency, never used as features)
    "trend_pct", "trend_direction", "is_declining_label",
]

lane_df = df[lane_cols].copy()
print("Unit of analysis: one row = one content page")
print("Shape:", lane_df.shape)
lane_df.head(8)

Unit of analysis: one row = one content page
Shape: (30000, 18)


,content_id,client_id,content_type,main_intent,word_count,content_age_days,age_tier,days_since_last_update,freshness_tier,avg_position,position_tier,search_volume,competition_level,engagement_rate,scroll_rate,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3221.0,187,181-365,20,0-30,10.6,striking,10.0,HIGH,5.88,4.55,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,2481.0,445,365+,25,0-30,20.3,page_3_5,90.0,LOW,0.00,10.00,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,3515.0,141,91-180,20,0-30,36.5,page_3_5,0.0,LOW,0.00,28.57,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,NaN,463,365+,22,0-30,6.2,page_1,10.0,LOW,1.28,3.45,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,2803.0,263,181-365,14,0-30,44.0,page_3_5,0.0,LOW,0.00,24.29,-34.7,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3080.0,147,91-180,20,0-30,8.5,page_1,720.0,HIGH,0.00,25.00,-38.9,down,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,3059.0,90,31-90,20,0-30,7.0,page_1,0.0,LOW,0.00,0.00,-92.3,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,NaN,445,365+,22,0-30,21.2,page_3_5,590.0,MEDIUM,3.57,7.14,0.6,stable,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

***Answer:***

No single column cleanly separates "declining" from "not declining" — the decline rate barely
moves when I slice by any one attribute on its own (see the code cell below): freshness, age, and
content type each shift the rate by only a few points around the ~54% base rate, and even the
biggest single-column gap I found (content_type) still leaves both groups far from 0% or 100%.
That means a plain if-statement rule (e.g. "flag anything not updated in 90 days," or "flag
anything older than a year") would misclassify a large share of pages either way — it's picking up
a real but weak signal, not a clean split. A model that combines several of these weak, tangled
signals together (age + freshness + position + content type + engagement, jointly) can separate
the groups far better than any one of them alone, which is exactly the case where ML earns its
place over a hand-written rule.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
for col in ["freshness_tier", "age_tier", "content_type"]:
    print(f"Decline rate by {col}:")
    print((df.groupby(col)["is_declining_label"].mean() * 100).round(1))
    print()

Decline rate by freshness_tier:
freshness_tier
0-30      51.1
181+      47.1
31-90     58.9
91-180    61.1
Name: is_declining_label, dtype: float64

Decline rate by age_tier:
age_tier
181-365    51.5
31-90      66.9
365+       42.6
91-180     62.6
Name: is_declining_label, dtype: float64

Decline rate by content_type:
content_type
comparison article    57.2
feedly article        28.7
keyword article       56.1
Name: is_declining_label, dtype: float64



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.